In [ ]:
# %% 셀 1: 데이터 로드 + 임베딩 로드 + STT 로드 + train/eval 분리
import os, json, random
import numpy as np
import torch
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
CELL_SIZE_X = 9
CELL_SIZE_Y = 16
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None

    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])

        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f

    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))

    video_name = data.get("video", "")

    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))

        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })

    # STT 로드
    fname = os.path.basename(path)[:-5]
    stt_path = os.path.join(STT_DIR, channel, fname + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass

    return {
        "channel": channel,
        "video_name": video_name,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


# ── 파일 목록 수집 ──
json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

print(f"📁 파일 수: {len(json_paths):,}개")

# ── 병렬 로드 ──
channel_set = set()
samples = []

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# ── 통계 ──
n_with_inst = sum(1 for s in samples if len(s["instances"]) > 0)
n_without_inst = sum(1 for s in samples if len(s["instances"]) == 0)
stt_counts = np.array([len(s["stt_segments"]) for s in samples])
inst_counts = np.array([len(s["instances"]) for s in samples])

print(f"\n📊 영상별 STT 세그먼트 수")
print(f"  mean: {stt_counts.mean():.1f}  median: {np.median(stt_counts):.0f}  "
      f"p99: {np.percentile(stt_counts, 99):.0f}  max: {stt_counts.max()}")

# ── train/eval 분리 ──
rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples, eval_samples = [], []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["video_name"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

print(f"\n✅ 영상: {len(samples):,}개 (텔롭 있음: {n_with_inst:,}  없음: {n_without_inst:,})  |  채널: {len(channels)}개")
print(f"✅ train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")
print(f"📊 인스턴스 수 — mean: {inst_counts.mean():.0f}  p99: {np.percentile(inst_counts, 99):.0f}  max: {inst_counts.max()}")

In [ ]:
'''
# %% POS_WEIGHT 계산 (per-slot mask 기준)
from tqdm import tqdm

total_pos = 0
total_pixels = 0

for s in tqdm(train_samples, desc="POS_WEIGHT 계산"):
    instances = s["instances"]
    if len(instances) == 0:
        continue

    for inst in instances:
        cx, cy = int(inst["x"]), int(inst["y"])
        w, h   = int(inst["w"]), int(inst["h"])
        x0 = max(0, cx - w // 2)
        y0 = max(0, cy - h // 2)
        x1 = min(GRID_W, x0 + w)
        y1 = min(GRID_H, y0 + h)

        if x1 > x0 and y1 > y0:
            pos = (x1 - x0) * (y1 - y0)
        else:
            pos = 0

        total_pos += pos
        total_pixels += GRID_H * GRID_W

pos_ratio = total_pos / total_pixels
new_pos_weight = (1 - pos_ratio) / pos_ratio

print(f"📊 per-slot mask 통계")
print(f"   positive 비율: {pos_ratio:.4f} ({pos_ratio*100:.2f}%)")
print(f"   POS_WEIGHT: {new_pos_weight:.1f}")
'''

In [ ]:
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

print(f"\n🔬 샘플링: {len(channels)}개 채널")
print(f"   train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")

In [ ]:
# %% 셀 2: Dataset + DataLoader (slot attention용)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 8
NUM_WORKERS_DL = 8
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5
TIME_DIM = 3
PATCH_SIZE = 8
N_PATCHES_X = GRID_W // PATCH_SIZE
N_PATCHES_Y = GRID_H // PATCH_SIZE
N_PATCHES = N_PATCHES_X * N_PATCHES_Y


class FrameBBoxDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = [s for s in samples if len(s["instances"]) > 0]
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        channel_id = self.channel2id[s["channel"]]
        instances = s["instances"]
        duration = max(s["duration"], 0.1)
        n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))
        n_inst = len(instances)

        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances])
        inst_cy = np.array([inst["y"] for inst in instances])
        inst_w  = np.array([inst["w"] for inst in instances])
        inst_h  = np.array([inst["h"] for inst in instances])

        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)
        video_emb   = self.text2emb.get(s["video_name"], ZERO_EMB)
        inst_embs = torch.stack([
            self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances
        ])
        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)
        stt_segments = s["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs   = torch.stack([
                self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments
            ])
            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0),
                        stt_embs[stt_active_idx[0]].unsqueeze(0),
                    ).item()
                    has_stts[i] = 1.0

        times = np.arange(n_frames, dtype=np.float32) / FPS
        active_matrix = (
            (inst_starts[None, :] <= times[:, None] + 0.05) &
            (inst_ends[None, :]   >  times[:, None])
        )

        telop_feats   = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, SLOT_DIM), dtype=np.float32)
        telop_mask    = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
        gt_bboxes     = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, 4), dtype=np.float32)
        slot_inst_idx = np.full((n_frames, MAX_ACTIVE_PER_FRAME), -1, dtype=np.int64)

        for fi in range(n_frames):
            active_idx = np.where(active_matrix[fi])[0]
            if len(active_idx) > 0:
                sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
                sorted_idx = active_idx[sorted_order]
                for si, ai in enumerate(sorted_idx):
                    telop_feats[fi, si, 0] = inst_tlens[ai] / MAX_TEXT_LEN
                    telop_feats[fi, si, 1] = ch_sims[ai]
                    telop_feats[fi, si, 2] = vid_sims[ai]
                    telop_feats[fi, si, 3] = stt_sims[ai]
                    telop_feats[fi, si, 4] = has_stts[ai]
                    telop_mask[fi, si] = True
                    gt_bboxes[fi, si, 0] = inst_cx[ai]
                    gt_bboxes[fi, si, 1] = inst_cy[ai]
                    gt_bboxes[fi, si, 2] = inst_w[ai]
                    gt_bboxes[fi, si, 3] = inst_h[ai]
                    slot_inst_idx[fi, si] = ai

        time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
        t_norm = times / max(duration, 1.0)
        time_feats[:, 0] = t_norm
        time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
        time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

        return {
            "channel_id":    torch.tensor(channel_id, dtype=torch.long),
            "telop_feats":   torch.from_numpy(telop_feats),
            "telop_mask":    torch.from_numpy(telop_mask),
            "time_feats":    torch.from_numpy(time_feats),
            "gt_bboxes":     torch.from_numpy(gt_bboxes),
            "slot_inst_idx": torch.from_numpy(slot_inst_idx),
            "inst_embs":     inst_embs,
            "n_frames":      n_frames,
        }


def collate_fn(batch):
    max_T = max(b["n_frames"] for b in batch)
    max_inst = max(b["inst_embs"].shape[0] for b in batch)
    B = len(batch)

    channel_ids   = torch.stack([b["channel_id"] for b in batch])
    telop_feats   = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, SLOT_DIM)
    telop_mask    = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, dtype=torch.bool)
    time_feats    = torch.zeros(B, max_T, TIME_DIM)
    gt_bboxes     = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, 4)
    frame_mask    = torch.zeros(B, max_T, dtype=torch.bool)
    slot_inst_idx = torch.full((B, max_T, MAX_ACTIVE_PER_FRAME), -1, dtype=torch.long)
    inst_embs     = torch.zeros(B, max_inst, EMB_DIM)

    for i, b in enumerate(batch):
        T = b["n_frames"]
        n_inst = b["inst_embs"].shape[0]
        telop_feats[i, :T]   = b["telop_feats"]
        telop_mask[i, :T]    = b["telop_mask"]
        time_feats[i, :T]    = b["time_feats"]
        gt_bboxes[i, :T]     = b["gt_bboxes"]
        frame_mask[i, :T]    = True
        slot_inst_idx[i, :T] = b["slot_inst_idx"]
        inst_embs[i, :n_inst] = b["inst_embs"]

    return {
        "channel_ids":   channel_ids,
        "telop_feats":   telop_feats,
        "telop_mask":    telop_mask,
        "time_feats":    time_feats,
        "gt_bboxes":     gt_bboxes,
        "frame_mask":    frame_mask,
        "slot_inst_idx": slot_inst_idx,
        "inst_embs":     inst_embs,
    }


train_ds = FrameBBoxDataset(train_samples, channel2id, text2emb)
eval_ds  = FrameBBoxDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn, drop_last=True,
    persistent_workers=True,
)
eval_loader = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn,
    persistent_workers=True,
)

print(f"✅ Dataset: train={len(train_ds):,}  eval={len(eval_ds):,}")
print(f"   EMB_DIM={EMB_DIM}  SLOT_DIM={SLOT_DIM}  BATCH_SIZE={BATCH_SIZE}")

batch = next(iter(train_loader))
print(f"\n✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

In [ ]:
# %% 셀 3: SlotSelfBBox — 슬롯 간 self-attention으로 위치 학습 (merged 없음)
D_MODEL = 256
N_HEADS = 8
N_LAYERS_SLOT = 6
D_FF = 512
DROPOUT = 0.1
INTRA_CHUNK = 512


class SlotSelfBBox(nn.Module):
    def __init__(self, n_channels, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers_slot=N_LAYERS_SLOT,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        self.emb_proj = nn.Sequential(
            nn.Linear(EMB_DIM, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.channel_emb = nn.Embedding(n_channels, d_model)
        self.time_proj = nn.Sequential(
            nn.Linear(TIME_DIM, d_model),
            nn.GELU(),
        )
        self.slot_pos_emb = nn.Embedding(MAX_ACTIVE_PER_FRAME, d_model)
        nn.init.normal_(self.slot_pos_emb.weight, mean=0.0, std=0.02)

        self.slot_norm = nn.LayerNorm(d_model)
        sl = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.slot_transformer = nn.TransformerEncoder(
            sl, num_layers=n_layers_slot, enable_nested_tensor=False,
        )

        self.bbox_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 4),
        )

    def forward(self, channel_ids, telop_mask, time_feats, frame_mask,
                slot_inst_idx, inst_embs):
        B, T, K = telop_mask.shape
        device = telop_mask.device
        dtype = inst_embs.dtype
        BT = B * T

        smask_flat = telop_mask.reshape(BT, K)
        frame_mask_flat = frame_mask.reshape(BT)

        # ── 슬롯 token 구성: text_emb + channel + time + pos ──
        idx_flat = slot_inst_idx.reshape(BT, K).clamp(min=0)
        batch_idx = torch.arange(BT, device=device) // T
        batch_exp = batch_idx.unsqueeze(1).expand(-1, K)
        slot_embs = inst_embs[batch_exp, idx_flat]  # (BT, K, EMB_DIM)

        ch = self.channel_emb(channel_ids)
        ch_exp = ch.unsqueeze(1).expand(-1, T, -1).reshape(BT, 1, -1)  # (BT, 1, d_model)

        time_flat = time_feats.reshape(BT, TIME_DIM)
        time_enc = self.time_proj(time_flat).unsqueeze(1)  # (BT, 1, d_model)

        slot_idx = torch.arange(K, device=device)

        slot_tokens = self.emb_proj(slot_embs) + ch_exp + time_enc + \
                      self.slot_pos_emb(slot_idx).unsqueeze(0)  # (BT, K, d_model)

        # ── 유효 프레임만 처리 ──
        valid_frame_idx = (frame_mask_flat & smask_flat.any(dim=1)).nonzero(as_tuple=True)[0]
        n_valid = valid_frame_idx.shape[0]

        all_slots = torch.zeros(BT, K, slot_tokens.shape[-1], device=device, dtype=dtype)

        if n_valid > 0:
            valid_tokens = self.slot_norm(slot_tokens[valid_frame_idx])
            valid_smask = smask_flat[valid_frame_idx]

            out_list = []
            for s in range(0, n_valid, INTRA_CHUNK):
                e = min(s + INTRA_CHUNK, n_valid)
                chunk = valid_tokens[s:e]
                mask_chunk = valid_smask[s:e]
                out = self.slot_transformer(
                    chunk, src_key_padding_mask=~mask_chunk,
                )
                out_list.append(out)
            all_updated = torch.cat(out_list, dim=0)
            all_slots[valid_frame_idx] = all_updated.to(dtype)

        # ── active slot 추출 → bbox 예측 ──
        active_combined = smask_flat & frame_mask_flat.unsqueeze(-1)
        active_idx = active_combined.nonzero(as_tuple=False)

        if active_idx.shape[0] > 0:
            active_slots = all_slots[active_idx[:, 0], active_idx[:, 1]]

            raw = self.bbox_head(active_slots)
            pred_cx = torch.sigmoid(raw[:, 0]) * GRID_W
            pred_cy = torch.sigmoid(raw[:, 1]) * GRID_H
            pred_w  = torch.sigmoid(raw[:, 2]) * GRID_W
            pred_h  = torch.sigmoid(raw[:, 3]) * GRID_H
            pred_bboxes = torch.stack([pred_cx, pred_cy, pred_w, pred_h], dim=-1)
        else:
            pred_bboxes = torch.zeros(0, 4, device=device, dtype=dtype)

        return pred_bboxes, active_idx


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SlotSelfBBox(n_channels=len(channels)).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")
print(f"   text_emb + channel + time + pos → TransformerEncoder {N_LAYERS_SLOT}L → bbox")
print(f"   d_model={D_MODEL}  n_heads={N_HEADS}  d_ff={D_FF}")
print(f"   merged 모델 불필요")

In [ ]:
# %% 셀 4: 학습 (SlotSelfBBox — GIoU + Smooth L1)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 20
LR = 1e-4
SAVE_DIR = "./model/8_slot_self_bbox"
os.makedirs(SAVE_DIR, exist_ok=True)

BBOX_SCALE = torch.tensor([GRID_W, GRID_H, GRID_W, GRID_H], dtype=torch.float32)


def giou_loss(pred, gt):
    pred_x1 = pred[:, 0] - pred[:, 2] / 2
    pred_y1 = pred[:, 1] - pred[:, 3] / 2
    pred_x2 = pred[:, 0] + pred[:, 2] / 2
    pred_y2 = pred[:, 1] + pred[:, 3] / 2

    gt_x1 = gt[:, 0] - gt[:, 2] / 2
    gt_y1 = gt[:, 1] - gt[:, 3] / 2
    gt_x2 = gt[:, 0] + gt[:, 2] / 2
    gt_y2 = gt[:, 1] + gt[:, 3] / 2

    inter_x1 = torch.max(pred_x1, gt_x1)
    inter_y1 = torch.max(pred_y1, gt_y1)
    inter_x2 = torch.min(pred_x2, gt_x2)
    inter_y2 = torch.min(pred_y2, gt_y2)
    inter_area = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)

    pred_area = (pred_x2 - pred_x1).clamp(0) * (pred_y2 - pred_y1).clamp(0)
    gt_area = (gt_x2 - gt_x1).clamp(0) * (gt_y2 - gt_y1).clamp(0)
    union_area = pred_area + gt_area - inter_area
    iou = inter_area / (union_area + 1e-8)

    enc_x1 = torch.min(pred_x1, gt_x1)
    enc_y1 = torch.min(pred_y1, gt_y1)
    enc_x2 = torch.max(pred_x2, gt_x2)
    enc_y2 = torch.max(pred_y2, gt_y2)
    enc_area = (enc_x2 - enc_x1).clamp(0) * (enc_y2 - enc_y1).clamp(0)

    giou = iou - (enc_area - union_area) / (enc_area + 1e-8)
    return (1 - giou).mean(), iou


def bbox_loss(pred, gt):
    scale = BBOX_SCALE.to(pred.device)
    l1 = F.smooth_l1_loss(pred / scale, gt / scale)
    g_loss, iou = giou_loss(pred, gt)
    return l1 + g_loss, iou


@torch.no_grad()
def compute_bbox_metrics(pred, gt):
    if pred.shape[0] == 0:
        return None
    _, iou = giou_loss(pred, gt)
    center_dist = torch.sqrt(
        (pred[:, 0] - gt[:, 0]) ** 2 + (pred[:, 1] - gt[:, 1]) ** 2 + 1e-8
    )
    w_err = (pred[:, 2] - gt[:, 2]).abs()
    h_err = (pred[:, 3] - gt[:, 3]).abs()
    return {
        "IoU": iou.mean().item(),
        "IoU50": (iou > 0.5).float().mean().item(),
        "IoU75": (iou > 0.75).float().mean().item(),
        "cL2": center_dist.mean().item(),
        "wMAE": w_err.mean().item(),
        "hMAE": h_err.mean().item(),
    }


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0.0, 0

    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        telop_mask    = batch["telop_mask"].to(device)
        gt_bboxes     = batch["gt_bboxes"].to(device)
        time_feats    = batch["time_feats"].to(device)
        frame_mask    = batch["frame_mask"].to(device)
        channel_ids   = batch["channel_ids"].to(device)
        slot_inst_idx = batch["slot_inst_idx"].to(device)
        inst_embs     = batch["inst_embs"].to(device)

        pred_bboxes, active_idx = model(
            channel_ids, telop_mask, time_feats, frame_mask,
            slot_inst_idx, inst_embs,
        )

        if pred_bboxes.shape[0] > 0:
            B, T = telop_mask.shape[:2]
            BT = B * T
            gt_flat = gt_bboxes.reshape(BT, MAX_ACTIVE_PER_FRAME, 4)
            active_gt = gt_flat[active_idx[:, 0], active_idx[:, 1]]
            loss, _ = bbox_loss(pred_bboxes.float(), active_gt.float())
        else:
            loss = torch.tensor(0.0, device=device, requires_grad=True)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    model.eval()
    eval_loss_sum, eval_batches = 0.0, 0
    all_metrics = []

    with torch.no_grad():
        for batch in eval_loader:
            telop_mask    = batch["telop_mask"].to(device)
            gt_bboxes     = batch["gt_bboxes"].to(device)
            time_feats    = batch["time_feats"].to(device)
            frame_mask    = batch["frame_mask"].to(device)
            channel_ids   = batch["channel_ids"].to(device)
            slot_inst_idx = batch["slot_inst_idx"].to(device)
            inst_embs     = batch["inst_embs"].to(device)

            pred_bboxes, active_idx = model(
                channel_ids, telop_mask, time_feats, frame_mask,
                slot_inst_idx, inst_embs,
            )

            if pred_bboxes.shape[0] > 0:
                B, T = telop_mask.shape[:2]
                BT = B * T
                gt_flat = gt_bboxes.reshape(BT, MAX_ACTIVE_PER_FRAME, 4)
                active_gt = gt_flat[active_idx[:, 0], active_idx[:, 1]]
                loss, _ = bbox_loss(pred_bboxes.float(), active_gt.float())
                m = compute_bbox_metrics(pred_bboxes.float(), active_gt.float())
                if m is not None:
                    all_metrics.append(m)
            else:
                loss = torch.tensor(0.0, device=device)

            eval_loss_sum += loss.item()
            eval_batches += 1

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss  = eval_loss_sum / max(eval_batches, 1)
    lr_now = optimizer.param_groups[0]["lr"]

    if all_metrics:
        avg_m = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0]}
    else:
        avg_m = {"IoU": 0, "IoU50": 0, "IoU75": 0, "cL2": 0, "wMAE": 0, "hMAE": 0}

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"IoU={avg_m['IoU']:.3f}  "
        f"IoU50={avg_m['IoU50']:.3f}  IoU75={avg_m['IoU75']:.3f}  "
        f"cL2={avg_m['cL2']:.1f}  "
        f"wMAE={avg_m['wMAE']:.1f}  hMAE={avg_m['hMAE']:.1f}  "
        f"lr={lr_now:.2e}"
    )

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

                                                               
[  1/20]  train=1.0702  eval=1.0566  IoU=0.044  IoU50=0.001  IoU75=0.000  cL2=22.4  wMAE=31.2  hMAE=46.4  lr=9.94e-05
   💾 best 갱신 (eval_loss=1.0566)
                                                               
[  2/20]  train=1.0524  eval=1.0508  IoU=0.052  IoU50=0.010  IoU75=0.002  cL2=21.8  wMAE=29.2  hMAE=41.0  lr=9.76e-05
   💾 best 갱신 (eval_loss=1.0508)
                                                               
[  3/20]  train=1.0451  eval=1.0371  IoU=0.062  IoU50=0.017  IoU75=0.002  cL2=21.4  wMAE=29.2  hMAE=42.3  lr=9.46e-05
   💾 best 갱신 (eval_loss=1.0371)
                                                               
[  4/20]  train=1.0418  eval=1.0378  IoU=0.062  IoU50=0.007  IoU75=0.001  cL2=21.3  wMAE=28.2  hMAE=42.6  lr=9.05e-05
[5/20] train:  85%|████████▍ | 590/696 [02:03<00:21,  4.83it/s]